# DM4CT Single Run In Colab

This notebook runs one **DM4CT** CT reconstruction experiment in Colab using the DM4CT codebase on the same small **TIFF** CT subset used by our PDHG notebook.

It is designed to make the cleanest possible side-by-side comparison:
- same CT test slices
- same number of views
- same medical checkpoint family
- explicit regime choice: `40_noiseless`, `80_noiseless`, or `80_noisy_10k`

By default it uses the repo-safe TIFF subset in `demo-samples/ct_l067_subset_tiff`.


## Runtime

In Colab, go to `Runtime > Change runtime type` and choose:
- `Python 3`
- `GPU`

Prefer `A100` when available.


In [ ]:
#@title Project Settings

OUR_REPO_URL = "https://github.com/Seif-Hussein/dyscode.git"  #@param {type:"string"}
OUR_REPO_BRANCH = "codex-pdhg-colab-light-100"  #@param {type:"string"}
DM4CT_REPO_URL = "https://github.com/DM4CT/DM4CT.git"  #@param {type:"string"}
DM4CT_REPO_BRANCH = "main"  #@param {type:"string"}

OUR_REPO_DIR = "/content/mycode2"  #@param {type:"string"}
DM4CT_REPO_DIR = "/content/DM4CT"  #@param {type:"string"}
PYTHON_BIN = "/usr/bin/python3"  #@param {type:"string"}
DRIVE_EXPORT_DIR = "/content/drive/MyDrive/dm4ct_single_run_exports"  #@param {type:"string"}
DRIVE_CT_DATA_DIR = ""  #@param {type:"string"}
DRIVE_CHECKPOINT_DIR = ""  #@param {type:"string"}
HF_MODEL_ID = "jiayangshi/lodochallenge_pixel_diffusion"  #@param {type:"string"}

RUN_NAME = "DM4CT_Single_Run"  #@param {type:"string"}
SESSION_TAG = ""  #@param {type:"string"}

DM4CT_METHOD = "DPS"  #@param ["DPS", "MCG", "PGDM", "DMPlug", "RedDiff", "HybridReg"]
DM4CT_REGIME = "80_noiseless"  #@param ["40_noiseless", "80_noiseless", "80_noisy_10k"]

DATA_START_IDX = 1  #@param {type:"integer"}
DATA_END_IDX = 2  #@param {type:"integer"}

SEED = 99  #@param {type:"integer"}
NUM_DETECTORS = 512  #@param {type:"integer"}
DPS_SCALE = 10.0  #@param {type:"number"}
MCG_SCALE = 2.0  #@param {type:"number"}
PGDM_SCALE = 0.3  #@param {type:"number"}
DMPLUG_EPOCHS = 1000  #@param {type:"integer"}
DMPLUG_LR = 0.005  #@param {type:"number"}
REDDIFF_STEPS = 200  #@param {type:"integer"}
HYBRIDREG_STEPS = 200  #@param {type:"integer"}
LOG_TAIL_LINES = 120  #@param {type:"integer"}


In [ ]:
#@title Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
#@title Fetch Repos
import os
import shutil
import subprocess
from pathlib import Path

for repo_dir, repo_url, repo_branch in [
    (OUR_REPO_DIR, OUR_REPO_URL, OUR_REPO_BRANCH),
    (DM4CT_REPO_DIR, DM4CT_REPO_URL, DM4CT_REPO_BRANCH),
]:
    path = Path(repo_dir)
    if path.exists():
        shutil.rmtree(path)
    subprocess.run([
        'git', 'clone', '--depth', '1', '--branch', repo_branch, repo_url, repo_dir
    ], check=True)

os.chdir(OUR_REPO_DIR)
subprocess.run(['git', 'status', '--short'], check=True)
os.chdir(DM4CT_REPO_DIR)
subprocess.run(['git', 'status', '--short'], check=True)


In [ ]:
#@title Install Dependencies
import os
import subprocess

subprocess.run([PYTHON_BIN, '-m', 'pip', 'install', '-r', f'{OUR_REPO_DIR}/requirements-colab-ct.txt'], check=True)
subprocess.run([PYTHON_BIN, '-m', 'pip', 'install', 'astra-toolbox', 'tifffile', 'scikit-image'], check=True)


In [ ]:
#@title Optional: Copy Local Checkpoint From Drive
import shutil
from pathlib import Path

local_checkpoint_path = None
if DRIVE_CHECKPOINT_DIR.strip():
    src = Path(DRIVE_CHECKPOINT_DIR)
    if not src.exists():
        raise FileNotFoundError(f'Checkpoint folder not found: {src}')
    dst = Path('/content') / 'dm4ct_checkpoint' / 'lodochallenge_pixel_diffusion'
    if dst.exists():
        shutil.rmtree(dst)
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(src, dst)
    local_checkpoint_path = dst.as_posix()
    print(f'Copied checkpoint to: {dst}')
else:
    print(f'Using Hugging Face checkpoint: {HF_MODEL_ID}')


In [ ]:
#@title Prepare Input Slice Folder
import shutil
from pathlib import Path

if DRIVE_CT_DATA_DIR.strip():
    source_dir = Path(DRIVE_CT_DATA_DIR)
else:
    source_dir = Path(OUR_REPO_DIR) / 'demo-samples' / 'ct_l067_subset_tiff'

if not source_dir.exists():
    raise FileNotFoundError(f'CT data folder not found: {source_dir}')

all_tifs = sorted([p for p in source_dir.iterdir() if p.suffix.lower() in {'.tif', '.tiff'}])
selected = all_tifs[DATA_START_IDX:DATA_END_IDX]
if not selected:
    raise RuntimeError('No TIFF slices selected. Check DATA_START_IDX/DATA_END_IDX.')

run_input_dir = Path('/content/dm4ct_single_input')
if run_input_dir.exists():
    shutil.rmtree(run_input_dir)
run_input_dir.mkdir(parents=True, exist_ok=True)
for src in selected:
    shutil.copy2(src, run_input_dir / src.name)

print('Source dir:', source_dir)
print('Selected files:', [p.name for p in selected])
print('Run input dir:', run_input_dir)


In [ ]:
#@title Write DM4CT Runner Script
from pathlib import Path

runner = Path('/content/run_dm4ct_single.py')
runner.write_text(r'''
import json
import math
import os
import random
import sys
from pathlib import Path

import astra
import numpy as np
import torch
from diffusers import DDPMScheduler, DDIMScheduler, UNet2DModel
from skimage.metrics import peak_signal_noise_ratio, structural_similarity
from tifffile import imread, imwrite
from torchvision import transforms

sys.path.insert(0, os.environ['DM4CT_REPO_DIR'])
from datasets import TiffDataset
from forward_operators_ct import Operator, NoNoise, PoissonNoise
from pipelines import (
    DDPMPipelineDPS,
    DDPMPipelineMCG,
    DDPMPipelinePGDM,
    DDPMPipelineDMPlug,
    DDPMPipelineRedDiff,
    DDPMPipelineHybridReg,
)
from condition_methods import (
    PosteriorSampling,
    ManifoldConstraintGradient,
    PGDM,
    DMPlug,
    RedDiff,
    HybridReg,
)

cfg = json.loads(Path(os.environ['DM4CT_RUN_CONFIG']).read_text(encoding='utf-8'))
seed = int(cfg['seed'])
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

transform = transforms.Compose([transforms.ToTensor()])
dataset = TiffDataset(cfg['data_path'], transform=transform)
num_angles = int(cfg['num_angles'])
angles = np.linspace(0, np.pi, num_angles)
vol_geom = astra.create_vol_geom(512, 512, 1)
proj_geom = astra.create_proj_geom('parallel3d', 1, 1, 1, int(cfg['num_detectors']), angles)
A = Operator(volume_geometry=vol_geom, projection_geometry=proj_geom)

regime = cfg['regime']
if regime.endswith('noiseless'):
    noiser = NoNoise()
else:
    noiser = PoissonNoise(0.5, float(cfg['photon_count']))

unet = UNet2DModel.from_pretrained(cfg['model_id']).to('cuda')
method = cfg['method']

if method == 'DPS':
    scheduler = DDPMScheduler(num_train_timesteps=1000)
    pipeline = DDPMPipelineDPS(unet=unet, scheduler=scheduler)
    conditioning_method = PosteriorSampling(operator=A, noiser=noiser, scale=float(cfg['dps_scale']))
elif method == 'MCG':
    scheduler = DDPMScheduler(num_train_timesteps=1000)
    pipeline = DDPMPipelineMCG(unet=unet, scheduler=scheduler)
    conditioning_method = ManifoldConstraintGradient(operator=A, noiser=noiser, scale=float(cfg['mcg_scale']))
elif method == 'PGDM':
    scheduler = DDIMScheduler(num_train_timesteps=1000)
    pipeline = DDPMPipelinePGDM(unet=unet, scheduler=scheduler)
    conditioning_method = PGDM(operator=A, noiser=noiser)
elif method == 'DMPlug':
    scheduler = DDIMScheduler(num_train_timesteps=1000)
    pipeline = DDPMPipelineDMPlug(unet=unet, scheduler=scheduler)
    conditioning_method = DMPlug(operator=A, noiser=noiser)
elif method == 'RedDiff':
    scheduler = DDIMScheduler(num_train_timesteps=1000)
    pipeline = DDPMPipelineRedDiff(unet=unet, scheduler=scheduler)
    conditioning_method = RedDiff(operator=A, noiser=noiser)
elif method == 'HybridReg':
    scheduler = DDIMScheduler(num_train_timesteps=1000)
    pipeline = DDPMPipelineHybridReg(unet=unet, scheduler=scheduler)
    conditioning_method = HybridReg(operator=A, noiser=noiser)
else:
    raise ValueError(f'Unsupported method: {method}')

pipeline.measurement_condition = conditioning_method
save_root = Path(cfg['save_root'])
save_root.mkdir(parents=True, exist_ok=True)
recon_dir = save_root / 'recons'
recon_dir.mkdir(parents=True, exist_ok=True)

results = []
for i in range(len(dataset)):
    img = dataset[i]
    gt = np.asarray(img[0].cpu().numpy(), dtype=np.float32)
    img_cuda = torch.tensor(img, device='cuda')
    y = A(img_cuda)
    y_n = noiser(y)

    if method == 'DPS':
        recon = pipeline(num_inference_steps=1000, measurement=y_n, output_type=np.array).images[0]
    elif method == 'MCG':
        recon = pipeline(num_inference_steps=1000, measurement=y_n, output_type=np.array).images[0]
    elif method == 'PGDM':
        recon = pipeline(num_inference_steps=100, measurement=y_n, output_type=np.array, scale=float(cfg['pgdm_scale'])).images[0]
    elif method == 'DMPlug':
        recon = pipeline(num_inference_steps=3, measurement=y_n, epochs=int(cfg['dmplug_epochs']), lr=float(cfg['dmplug_lr']), output_type=np.array).images[0]
    elif method == 'RedDiff':
        recon = pipeline(num_inference_steps=int(cfg['reddiff_steps']), measurement=y_n, sigma=1e-4, loss_measurement_weight=0.5, loss_noise_weight=10000, lr=0.099, output_type=np.array).images[0]
    elif method == 'HybridReg':
        recon = pipeline(num_inference_steps=int(cfg['hybridreg_steps']), measurement=y_n, sigma=1e-4, loss_measurement_weight=0.5, loss_noise_weight=10000, lr=0.099, beta=0.9999, output_type=np.array).images[0]

    recon = np.asarray(recon, dtype=np.float32)
    if recon.ndim == 3:
        recon = recon[..., 0]
    recon = np.clip(recon, 0.0, 1.0)
    psnr = peak_signal_noise_ratio(gt, recon, data_range=1.0)
    ssim = structural_similarity(gt, recon, data_range=1.0)
    results.append({'index': i, 'psnr': float(psnr), 'ssim': float(ssim)})
    imwrite(recon_dir / f'{i:03d}.tif', recon)

summary = {
    'method': method,
    'regime': regime,
    'num_angles': num_angles,
    'num_detectors': int(cfg['num_detectors']),
    'photon_count': None if regime.endswith('noiseless') else float(cfg['photon_count']),
    'results': results,
    'avg_psnr': float(np.mean([r['psnr'] for r in results])),
    'avg_ssim': float(np.mean([r['ssim'] for r in results])),
}
(summary_path := save_root / 'metrics.json').write_text(json.dumps(summary, indent=2), encoding='utf-8')
print(json.dumps(summary, indent=2))
''', encoding='utf-8')
print(runner)


In [ ]:
#@title Build Run Command
import json
import os
import shlex
from datetime import datetime
from pathlib import Path

if DM4CT_REGIME == '40_noiseless':
    regime_num_angles = 40
    photon_count = None
elif DM4CT_REGIME == '80_noiseless':
    regime_num_angles = 80
    photon_count = None
elif DM4CT_REGIME == '80_noisy_10k':
    regime_num_angles = 80
    photon_count = 10000.0
else:
    raise ValueError(f'Unsupported DM4CT_REGIME: {DM4CT_REGIME}')

timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
session_bits = [RUN_NAME.strip(), DM4CT_METHOD.strip(), DM4CT_REGIME.strip(), SESSION_TAG.strip(), timestamp]
run_tag = '_'.join(bit for bit in session_bits if bit).replace(' ', '_')
save_root = Path('/content') / 'dm4ct_single_results' / run_tag
run_aux_root = save_root / 'run_aux'
latest_log_path = run_aux_root / 'run.log'
latest_pid_path = run_aux_root / 'run.pid'
config_path = run_aux_root / 'runner_config.json'
run_aux_root.mkdir(parents=True, exist_ok=True)

model_id = local_checkpoint_path if local_checkpoint_path else HF_MODEL_ID
runner_cfg = {
    'seed': SEED,
    'method': DM4CT_METHOD,
    'regime': DM4CT_REGIME,
    'num_angles': regime_num_angles,
    'num_detectors': NUM_DETECTORS,
    'photon_count': photon_count,
    'model_id': model_id,
    'data_path': '/content/dm4ct_single_input',
    'save_root': save_root.as_posix(),
    'dps_scale': DPS_SCALE,
    'mcg_scale': MCG_SCALE,
    'pgdm_scale': PGDM_SCALE,
    'dmplug_epochs': DMPLUG_EPOCHS,
    'dmplug_lr': DMPLUG_LR,
    'reddiff_steps': REDDIFF_STEPS,
    'hybridreg_steps': HYBRIDREG_STEPS,
}
config_path.write_text(json.dumps(runner_cfg, indent=2), encoding='utf-8')

run_cmd = [PYTHON_BIN, '/content/run_dm4ct_single.py']
print('Command:
')
print(' '.join(shlex.quote(part) for part in run_cmd))

last_context = {
    'run_tag': run_tag,
    'save_root': save_root.as_posix(),
    'latest_log_path': latest_log_path.as_posix(),
    'latest_pid_path': latest_pid_path.as_posix(),
    'runner_config_path': config_path.as_posix(),
    'run_cmd': run_cmd,
}
(config_path.parent / 'context.json').write_text(json.dumps(last_context, indent=2), encoding='utf-8')
print(f'Context saved to: {config_path.parent / "context.json"}')


In [ ]:
#@title Launch The Run In Background
import os
import subprocess
from pathlib import Path

env = os.environ.copy()
env['DM4CT_REPO_DIR'] = DM4CT_REPO_DIR
env['DM4CT_RUN_CONFIG'] = last_context['runner_config_path']

latest_log_path = Path(last_context['latest_log_path'])
latest_pid_path = Path(last_context['latest_pid_path'])
latest_log_path.parent.mkdir(parents=True, exist_ok=True)

with latest_log_path.open('w', encoding='utf-8') as log_handle:
    process = subprocess.Popen(
        last_context['run_cmd'],
        cwd='/content',
        env=env,
        stdout=log_handle,
        stderr=subprocess.STDOUT,
        text=True,
    )
latest_pid_path.write_text(str(process.pid), encoding='utf-8')
print(f'PID: {process.pid}')
print(f'Log: {latest_log_path}')
print(f'Save root: {last_context["save_root"]}')


In [ ]:
#@title Show Recent Log Lines
from pathlib import Path

log_path = Path(last_context['latest_log_path'])
if not log_path.exists():
    raise FileNotFoundError(f'Log file not found: {log_path}')
lines = log_path.read_text(encoding='utf-8', errors='ignore').splitlines()
tail = lines[-int(LOG_TAIL_LINES):]
print('
'.join(tail) if tail else '<log is empty>')


In [ ]:
#@title Show Results
import json
from pathlib import Path

save_root = Path(last_context['save_root'])
metrics_path = save_root / 'metrics.json'
recon_dir = save_root / 'recons'
print('save_root:', save_root)
print('recon_dir exists:', recon_dir.exists())
print('recon files:', [p.name for p in sorted(recon_dir.glob('*.tif'))] if recon_dir.exists() else [])
if metrics_path.exists():
    print(metrics_path.read_text(encoding='utf-8'))
else:
    print('No metrics.json found yet.')


In [ ]:
#@title Copy Run Artifacts To Drive
import shutil
from pathlib import Path

export_root = Path(DRIVE_EXPORT_DIR)
export_root.mkdir(parents=True, exist_ok=True)
for src in [Path(last_context['save_root']), Path(last_context['latest_log_path'])]:
    if not src.exists():
        print(f'Skipping missing path: {src}')
        continue
    dst = export_root / src.name
    if src.is_dir():
        if dst.exists():
            shutil.rmtree(dst)
        shutil.copytree(src, dst)
    else:
        shutil.copy2(src, dst)
    print(f'Copied {src} -> {dst}')
